# Build the Regression Model

### Setup
Recreate the cleaned dataset, train-test split and fitted preprocessor from the earlier notebooks, so this notebook can run on its own. Place `insurance.csv` and `validation_dataset.csv` in the same folder.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- Data cleaning (Section 2) ---
insurance = pd.read_csv('insurance.csv')
insurance_clean = insurance.copy()

insurance_clean['charges'] = pd.to_numeric(
    insurance_clean['charges'].astype(str).str.replace('$', '', regex=False),
    errors='coerce'
)
insurance_clean = insurance_clean.dropna(subset=['charges'])
insurance_clean = insurance_clean[(insurance_clean['age'] >= 0) | (insurance_clean['age'].isnull())]
insurance_clean = insurance_clean[(insurance_clean['children'] >= 0) | (insurance_clean['children'].isnull())]

sex_mapping = {'male': 'male', 'M': 'male', 'man': 'male',
               'female': 'female', 'F': 'female', 'woman': 'female'}
insurance_clean['sex'] = insurance_clean['sex'].map(sex_mapping)
insurance_clean['region'] = insurance_clean['region'].str.title()
insurance_clean = insurance_clean.drop_duplicates()

# --- Prepare data for modelling (Section 3) ---
X = insurance_clean.drop(columns=['charges'])
y = insurance_clean['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

## 4.1 Build a Linear Regression Model
Linear Regression is a supervised machine learning algorithm that predicts a numerical value by learning the relationship between input features and a target variable.

In this project, the model will learn how customer characteristics such as age, BMI, smoking status and region relate to insurance charges.

In [ ]:
# Step 1: Import Linear Regression
from sklearn.linear_model import LinearRegression

# Step 2: Initialize the model
linear_model = LinearRegression()

# Step 3: Train the model
linear_model.fit(X_train_processed, y_train)

# Step 4: Generate predictions
y_pred = linear_model.predict(X_test_processed)

print("First 10 predicted charges:")
print(y_pred[:10])

#### Step 3: Train the model
Understanding the code:
- `X_train_processed`: The 933 preprocessed training records containing 11 numerical features.
- `y_train`: The actual insurance charges for those 933 records.
- `.fit()`: Trains the model by learning the relationship between the input features and insurance charges.

#### Step 4: Generate predictions
Understanding the code:
- `linear_model.predict()`: Uses the trained model to generate predictions.
- `X_test_processed`: Contains the 234 testing records.
- `y_pred`: Stores the predicted insurance charges.

We will later compare these predictions against the actual charges in `y_test` to evaluate the model's performance.

### Interpretation
The trained Linear Regression model was used to predict insurance charges for the testing dataset. The model generated 234 predictions based on the 11 preprocessed features. The first 10 predictions were displayed to verify that the model could successfully produce numerical outputs. These predictions will be compared with the actual insurance charges to evaluate the model's performance.

**One observation:**
The model predicted RM493.03 for one record. This is below the project's minimum charge requirement of RM1,000. We will address this when generating the final validation predictions.

## 4.2 Evaluate the Model Performance
Now, we need to evaluate how accurately our Linear Regression model predicts insurance charges.

We will use three evaluation metrics:
1. **R-squared (R²)**: Measures how much variation in insurance charges is explained by the model.
2. **Mean Absolute Error (MAE)**: Measures the average absolute difference between actual and predicted charges.
3. **Root Mean Squared Error (RMSE)**: Measures the square root of the average squared prediction errors, giving more weight to larger errors.

### 4.2.1 Calculate R-squared (R²)
R-squared is particularly important because the project requires an R² score greater than 0.65.

In [ ]:
# Step 1: Import the evaluation metrics
from sklearn.metrics import r2_score as calculate_r2

# Step 2: Calculate the R² score
r2_value = calculate_r2(y_test, y_pred)
print("R-squared:", r2_value)

Understanding the code:
* `y_test`: Actual insurance charges from the testing dataset.
* `y_pred`: Insurance charges predicted by the Linear Regression model.
* `r2_score()`: Compares actual and predicted values to measure how well the model explains the variation in insurance charges.

#### How to interpret R²?
| R² value | Interpretation |
|---|---|
| 1.0 | Perfect predictions |
| 0.8 | Strong explanatory performance |
| 0.65 | Project's minimum target |
| 0.0 | No improvement over predicting the mean |
| Below 0 | Worse than predicting the mean |

> **Note:** R² is not a percentage of predictions that are exactly correct.

### Interpretation
The Linear Regression model achieved an R² score of 0.7621 on the testing dataset. This indicates that approximately 76.21% of the variation in insurance charges can be explained by the features used in the model. Since the score exceeds the project's minimum requirement of 0.65, the model meets the required R² performance target.

### 4.2.2 Calculate Mean Absolute Error (MAE)
Next, we will calculate the Mean Absolute Error to understand how far the model's predictions are from the actual insurance charges on average.

**Understanding MAE:**
MAE measures the average absolute difference between actual and predicted insurance charges.
For example, an MAE of RM3,000 means that the model's predictions differ from the actual charges by approximately RM3,000 on average.

In [ ]:
# Step 1: Import the evaluation metrics
from sklearn.metrics import mean_absolute_error

# Step 2: Calculate MAE
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error:", mae)

### Interpretation
The Linear Regression model achieved a Mean Absolute Error (MAE) of approximately RM4,037.32. This means that the model's predictions differ from the actual insurance charges by an average absolute amount of approximately RM4,037.32. A lower MAE indicates better prediction accuracy.

### 4.2.3 Calculate Root Mean Squared Error (RMSE)
Next, we will calculate RMSE to measure the prediction errors while giving greater weight to larger errors.

**Understanding RMSE:**
RMSE calculates the square root of the average squared differences between actual and predicted charges.
Unlike MAE, RMSE penalizes larger prediction errors more heavily. This helps us identify whether the model makes some predictions that are significantly different from the actual charges.

In [ ]:
# Step 1: Import the evaluation metrics
from sklearn.metrics import mean_squared_error

# Step 2: Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Root Mean Squared Error:", rmse)

### Interpretation
The Linear Regression model achieved a Root Mean Squared Error (RMSE) of approximately RM5,693.12. This indicates that the model has a relatively larger prediction error when greater weight is given to larger differences between actual and predicted insurance charges. Since RMSE is higher than MAE, it suggests that some predictions have larger errors than others.

### 4.2.4 Summarize Model Performance

In [ ]:
# Step 1: Create a performance summary
model_performance = pd.DataFrame({
    'Metric': ['R-squared', 'MAE', 'RMSE'],
    'Score': [r2_value, mae, rmse]
})

model_performance

### Interpretation
The Linear Regression model achieved an R² score of 0.7621, indicating that it explains approximately 76.21% of the variation in insurance charges. The model recorded an MAE of RM4,037.32 and an RMSE of RM5,693.12. The higher RMSE compared to MAE suggests that some predictions have relatively larger errors. Overall, the model meets the project's minimum R² requirement of 0.65 and provides a baseline for comparison with other regression models.

## 4.3 Evaluate Model Generalization Using Cross-Validation
Before moving to more advanced models, let's check whether the Linear Regression model performs consistently across different subsets of the training data.

We will use **5-Fold Cross-Validation**.

**What is 5-Fold Cross-Validation?**
The training dataset is divided into five equal parts (folds).
* Four folds are used for training.
* One fold is used for validation.
* This process repeats five times, with each fold serving as the validation set once.

This helps us understand whether the model's performance is consistent rather than depending on just one train-test split.

### 4.3.1 Calculate Cross-Validation Scores

In [ ]:
# Step 1: Import the required function
from sklearn.model_selection import cross_val_score

# Step 2: Calculate cross-validation scores
cv_scores = cross_val_score(
    linear_model,
    X_train_processed,
    y_train,
    cv=5,
    scoring='r2'
)

# Step 3: Calculate the average cross-validation score
cv_mean = cv_scores.mean()

# Step 4: Calculate the standard deviation
cv_std = cv_scores.std()

print("Cross-validation R² scores:", cv_scores)
print("Average Cross-validation R²:", cv_mean)
print("Cross-validation R² standard deviation:", cv_std)

**Understanding the parameters:**
| Parameter | Purpose |
|---|---|
| `linear_model` | The Linear Regression model being evaluated |
| `X_train_processed` | Preprocessed training features |
| `y_train` | Actual training insurance charges |
| `cv=5` | Divides the training data into five folds |
| `scoring='r2'` | Uses R-squared to evaluate each fold |

**Why calculate the average?**
The average R² score provides a more representative view of the model's performance across different training and validation subsets, rather than relying on a single fold.

**Why calculate standard deviation?**
* Lower standard deviation means the model performs more consistently across folds.
* Higher standard deviation means its performance varies more between folds.

> **Important note:** This evaluates the already-preprocessed training data. For a more rigorous cross-validation workflow, preprocessing should eventually be included inside the model pipeline so that imputation and scaling are refitted within each fold. We can improve this when comparing models.

### Interpretation
The Linear Regression model achieved an average cross-validation R² score of 0.7052 with a standard deviation of 0.0350. The relatively small standard deviation indicates that the model's performance does not vary significantly across the five folds. Although Fold 3 recorded a lower R² score of 0.6416, the remaining folds achieved scores above 0.70. Overall, the model demonstrates reasonably stable performance across different subsets of the training data.

### 4.3.2 Compare Testing and Cross-Validation Performance
| Evaluation | R² Score |
|---|---|
| Testing dataset | 0.7621 |
| Average cross-validation | 0.7052 |
| Difference | 0.0570 |

The testing score is approximately 0.0570 higher than the cross-validation average. This difference suggests some variation in performance depending on the data split, but it does not by itself establish overfitting.

**One important technical improvement before we continue:**
Our earlier cross-validation used data that had already been preprocessed using the full training set. This means preprocessing information was shared across folds. For a more reliable cross-validation result, we should eventually put the preprocessor and Linear Regression model into one Pipeline and repeat the evaluation.

## 4.4 Prepare the Validation Dataset
The validation dataset contains unseen records that we will use to generate insurance charge predictions.

First, load and inspect the dataset to understand its structure, columns, data types and missing values.

In [ ]:
# Step 1: Load the validation dataset
validation_data = pd.read_csv('validation_dataset.csv')

# Step 2: Check the dataset information
print("Validation dataset shape:", validation_data.shape)

print("\nDataset information:")
validation_data.info()

print("\nFirst 5 rows:")
display(validation_data.head())

print("\nMissing values:")
print(validation_data.isnull().sum())

> The validation dataset contains 50 records and 6 expected features, with no missing values. This means we can proceed directly to applying the existing preprocessing pipeline.

## 4.5 Prepare Validation Features

In [ ]:
# Step 1: Standardize categorical values
validation_data['sex'] = validation_data['sex'].map(sex_mapping)
validation_data['region'] = validation_data['region'].str.title()

# Step 2: Select the features
X_validation = validation_data[X.columns]

# Step 3: Apply the fitted preprocessing pipeline
X_validation_processed = preprocessor.transform(X_validation)

# Step 4: Check the processed dataset
print("Processed validation shape:", X_validation_processed.shape)
print("Missing values:", np.isnan(X_validation_processed).sum())

> This confirms that the validation data has the same 11 processed features as the training data.

## 4.6 Generate Predictions Using the Trained Model
Now, use the trained Linear Regression model to predict insurance charges for all 50 unseen records.

Any prediction below the project's minimum charge of RM1,000 is raised to RM1,000 using `np.maximum()`.

In [ ]:
# Step 1: Generate predictions
validation_predictions = linear_model.predict(X_validation_processed)

# Step 2: Apply the minimum charge (RM1,000)
validation_predictions = np.maximum(validation_predictions, 1000)

# Step 3: Display the first 10 predictions
print("First 10 predicted charges:")
print(validation_predictions[:10])

# Step 4: Check the prediction count
print("Number of predictions:", len(validation_predictions))

### Interpretation
The predictions now meet the minimum charge requirement:
* Number of predictions: 50
* Minimum predicted charge: RM1,000
* First prediction: RM1,000 (adjusted from RM588.73)

## 4.7 Store Predictions in the Validation Dataset
The final step is to add the predictions as a new column named `predicted_charges`.

In [ ]:
# Step 1: Create the new column
validation_data['predicted_charges'] = validation_predictions

# Step 2: Display the results
display(validation_data.head(10))

## Final Project Verification

In [ ]:
print("Final dataset shape:", validation_data.shape)
print("\nColumn names:")
print(validation_data.columns.tolist())

print("\nMinimum predicted charge:", validation_data['predicted_charges'].min())
print("Missing predictions:", validation_data['predicted_charges'].isnull().sum())
print("Number of predictions:", len(validation_data))
print("R-squared score:", r2_value)

### Save the Final Predictions
This saves all 50 predicted insurance charges into `insurance_predictions.csv`.

In [ ]:
validation_data.to_csv('insurance_predictions.csv', index=False)